In [1]:
%pip install transformers datasets accelerate gradio torch pandas -q


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import re
from transformers import (
    AutoTokenizer,
)

MODEL_NAME = "./distilgpt2_local"
OUTPUT_DIR = "./smart_compose_ckpt"
MAX_LENGTH = 128  # tokens per training sample
BATCH_SIZE = 8
EPOCHS = 3
LR = 5e-5
BEAM_WIDTH = 5  # beam search width (phrase generator)
MIN_CONF = 0.10  # drop suggestions below this score
MAX_SUGGESTION_WORDS = 10  # remove overly long suggestions

DEVICE = (
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
print(f"▶ Device : {DEVICE}")
print(f"▶ Model  : {MODEL_NAME}")

▶ Device : mps
▶ Model  : ./distilgpt2_local


In [3]:
import pandas as pd

train_df = pd.read_parquet("./data/train-00000-of-00001.parquet")
val_df = pd.read_parquet("./data/validation-00000-of-00001.parquet")
test_df = pd.read_parquet("./data/test-00000-of-00001.parquet")

raw = {
    "train": train_df,
    "validation": val_df,
    "test": test_df,
}

print(f"Train:      {len(train_df):,} rows")
print(f"Validation: {len(val_df):,} rows")
print(f"Test:       {len(test_df):,} rows")
print(f"\nColumns: {list(train_df.columns)}")

sample = raw["train"].iloc[0]
print("\n── Subject ──────────────────────────────")
print(sample["subject_line"])
print("── Body (first 300 chars) ───────────────")
print(sample["email_body"][:300])

Train:      14,436 rows
Validation: 1,960 rows
Test:       1,906 rows

Columns: ['email_body', 'subject_line']

── Subject ──────────────────────────────
Service Agreement
── Body (first 300 chars) ───────────────
Greg/Phillip,  Attached is the Grande Communications Service Agreement.
The business points can be found in Exhibit C.  I Can get the Non-Disturbance agreement after it has been executed by you and Grande.
I will fill in the Legal description of the property one I have received it.
Please execute an


In [4]:
from datasets import Dataset, DatasetDict


def clean_email(text: str) -> str:
    text = re.sub(r"-{3,}.*?-{3,}", " ", text, flags=re.DOTALL)
    text = re.sub(r"[\w.+-]+@[\w-]+\.[a-zA-Z]{2,}", "##EMAIL##", text)
    text = re.sub(r"\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b", "##PHONE##", text)
    text = re.sub(r"https?://\S+|www\.\S+", "##URL##", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()


def format_row(row):
    subj = str(row["subject_line"]).strip()
    body = " ".join(clean_email(str(row["email_body"])).split()[:200])
    return f"Subject: {subj}\n\n{body}<|endoftext|>"


# Apply with pandas, then convert to HuggingFace DatasetDict for the Trainer
cleaned = DatasetDict(
    {
        split: Dataset.from_dict({"text": df.apply(format_row, axis=1).tolist()})
        for split, df in raw.items()
    }
)

print(
    f"Total samples: {len(cleaned['train'])} train / {len(cleaned['validation'])} val"
)
print("\nFormatted sample:\n", cleaned["train"][0]["text"][:400])

Total samples: 14436 train / 1960 val

Formatted sample:
 Subject: Service Agreement

Greg/Phillip, Attached is the Grande Communications Service Agreement. The business points can be found in Exhibit C. I Can get the Non-Disturbance agreement after it has been executed by you and Grande. I will fill in the Legal description of the property one I have received it. Please execute and send to: Grande Communications, 401 Carlson Circle, San Marcos Texas, 78


In [5]:
import torch
from torch.utils.data import Dataset as TorchDataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token


def tokenize_split(ds):
    texts = [str(t) for t in ds["text"]]
    return tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors="pt",
    )


print("Tokenizing splits...", end=" ")
encodings = {split: tokenize_split(ds) for split, ds in cleaned.items()}
print("done ✅")

print(f"\nVocab size : {tokenizer.vocab_size:,}")
print(f"Train size : {encodings['train']['input_ids'].shape}")
print(f"Sample IDs : {encodings['train']['input_ids'][0][:20].tolist()}")


class EmailDataset(TorchDataset):
    def __init__(self, encodings):
        self.input_ids = encodings["input_ids"]
        self.attention_mask = encodings["attention_mask"]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.input_ids[idx].clone(),
        }


train_dataset = EmailDataset(encodings["train"])
val_dataset = EmailDataset(encodings["validation"])

print(f"\nTrain samples : {len(train_dataset):,}")
print(f"Val samples   : {len(val_dataset):,}")

Tokenizing splits... done ✅

Vocab size : 50,257
Train size : torch.Size([14436, 128])
Sample IDs : [19776, 25, 4809, 12729, 198, 198, 25025, 14, 41970, 541, 11, 3460, 2317, 318, 262, 29073, 14620, 4809, 12729, 13]

Train samples : 14,436
Val samples   : 1,960


In [6]:
# Cell 5 | Fine-tune — full fixed cell
import torch
import time
import math
import os
from transformers import (
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    TrainerCallback,
)

DEVICE = "cpu"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

# ── Download model if not already cached ─────────────────────────────────────
print("Checking cache...", flush=True)
try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, dtype=torch.float32, local_files_only=True
    )
    print("✅ Loaded from cache", flush=True)
except Exception:
    print("Not cached — downloading now (one time only)...", flush=True)
    os.environ["TRANSFORMERS_OFFLINE"] = "0"
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32)
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    print("✅ Downloaded and cached", flush=True)

model.train()
print(
    f"   {sum(p.numel() for p in model.parameters())/1e6:.1f}M params on {DEVICE}",
    flush=True,
)


# ── Live logging callback ─────────────────────────────────────────────────────
class LiveLogCallback(TrainerCallback):
    def __init__(self):
        self.t0 = time.time()

    def on_train_begin(self, args, state, control, **kwargs):
        print(f"\n{'Epoch':>6} {'Step':>7} {'Loss':>10} {'Elapsed':>10}")
        print("─" * 38)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        loss = logs.get("loss", logs.get("train_loss", 0))
        print(
            f"{state.epoch:>6.2f} {state.global_step:>7} {loss:>10.4f} {time.time()-self.t0:>9.1f}s",
            flush=True,
        )

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics:
            vl = metrics.get("eval_loss", 0)
            print(
                f"  ✦ val_loss={vl:.4f}  perplexity={math.exp(vl) if vl < 20 else float('inf'):.2f}"
            )
        print("─" * 38)


# ── Training args ─────────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_ratio=0.05,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=10,
    fp16=False,
    bf16=False,
    no_cuda=True,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    report_to="none",
)

# ── Trainer ───────────────────────────────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2),
        LiveLogCallback(),
    ],
)

print(
    f"\n📋 {len(train_dataset):,} train samples · {EPOCHS} epochs · batch {BATCH_SIZE}"
)
print(f"   ~{len(train_dataset)//BATCH_SIZE * EPOCHS:,} total steps")
print("🚀 Starting training...\n", flush=True)
trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\n✅ Model saved → {OUTPUT_DIR}")

Checking cache...
✅ Loaded from cache
   81.9M params on cpu

📋 14,436 train samples · 3 epochs · batch 8
   ~5,412 total steps
🚀 Starting training...


 Epoch    Step       Loss    Elapsed
──────────────────────────────────────


/Users/dalt353/GitHub/ML-Portfolio/.venv/lib/python3.10/site-packages/transformers/training_args.py:1636: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,2.805400,2.715422


  0.01      10     5.8857       5.3s
  0.01      20     6.0101      10.3s
  0.02      30     5.7109      15.3s
  0.02      40     5.0536      20.3s
  0.03      50     3.9742      25.3s
  0.03      60     3.4125      30.2s
  0.04      70     3.5055      35.2s
  0.04      80     3.2635      40.1s
  0.05      90     2.9823      45.1s
  0.06     100     3.1118      50.1s
  0.06     110     3.0978      55.1s
  0.07     120     3.1579      60.0s
  0.07     130     2.9279      65.0s
  0.08     140     2.9941      70.1s
  0.08     150     3.1500      75.3s
  0.09     160     2.8592      80.5s
  0.09     170     3.1057      85.6s
  0.10     180     3.0973      90.8s
  0.11     190     2.9606      95.9s
  0.11     200     3.0153     101.1s
  0.12     210     2.8225     106.2s
  0.12     220     3.1597     111.4s
  0.13     230     2.9912     116.6s
  0.13     240     2.9349     121.9s
  0.14     250     3.1281     127.0s
  0.14     260     2.9789     132.2s
  0.15     270     2.9095     137.4s
 

KeyboardInterrupt: 

In [ ]:
# ── Perplexity from eval loss ─────────────────────────────────────────────────
eval_results = trainer.evaluate()
perplexity = math.exp(eval_results["eval_loss"])
print(f"📊 Perplexity: {perplexity:.2f}  (lower is better)")


# ── ExactMatch@N ──────────────────────────────────────────────────────────────
def exact_match_at_n(prompts, ground_truths, tok, mdl, n=3):
    """Percentage of beam-search completions that match first N words of ground truth."""
    hits = 0
    for prompt, truth in zip(prompts, ground_truths):
        inputs = tok(prompt, return_tensors="pt").to(DEVICE)
        in_len = inputs["input_ids"].shape[1]
        with torch.no_grad():
            out = mdl.generate(
                **inputs,
                max_new_tokens=n + 3,
                num_beams=BEAM_WIDTH,
                early_stopping=True,
                pad_token_id=tok.eos_token_id,
            )
        generated = tok.decode(out[0][in_len:], skip_special_tokens=True)
        gen_words = generated.strip().split()[:n]
        true_words = truth.strip().split()[:n]
        if gen_words == true_words:
            hits += 1
    return hits / len(prompts)


# Use 100 validation samples for speed
val_samples = raw["validation"].select(range(100))
prompts = [
    f"Subject: {s['subject_line']}\n\n{clean_email(s['email_body'])[:80]}"
    for s in val_samples
]
truths = [clean_email(s["email_body"])[80:] for s in val_samples]

tok_eval, mdl_eval = tokenizer, model
for n in [3, 5]:
    score = exact_match_at_n(prompts, truths, tok_eval, mdl_eval, n=n)
    print(f"📊 ExactMatch@{n}: {score:.3f}")

In [ ]:
def load_finetuned(model_dir=OUTPUT_DIR):
    tok = AutoTokenizer.from_pretrained(model_dir)
    mdl = AutoModelForCausalLM.from_pretrained(model_dir).to(DEVICE)
    mdl.eval()
    return tok, mdl


def phrase_generator(
    prompt: str,
    tok,
    mdl,
    beam_width=BEAM_WIDTH,
    max_new=20,
    min_conf=MIN_CONF,
    max_words=MAX_SUGGESTION_WORDS,
) -> list[dict]:
    """
    Returns filtered suggestions as [{text, score}].
    Mirrors the Smart Compose phrase generator:
      1. Beam search → top-k completions
      2. Remove long suggestions
      3. Remove low-confidence suggestions
    """
    inputs = tok(prompt, return_tensors="pt").to(DEVICE)
    in_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = mdl.generate(
            **inputs,
            max_new_tokens=max_new,
            num_beams=beam_width,
            num_return_sequences=beam_width,
            early_stopping=True,
            output_scores=True,
            return_dict_in_generate=True,
            pad_token_id=tok.eos_token_id,
            eos_token_id=tok.eos_token_id,
            repetition_penalty=1.3,
        )

    suggestions = []
    for seq, score in zip(outputs.sequences, outputs.sequences_scores):
        text = tok.decode(seq[in_len:], skip_special_tokens=True)
        text = text.split("\n")[0].strip()  # single line only
        conf = score.exp().item()  # convert log-prob → prob

        # ── Filter 1: Remove long suggestions ────────────────────────────────
        if len(text.split()) > max_words:
            text = " ".join(text.split()[:max_words])

        # ── Filter 2: Remove low-confidence suggestions ───────────────────────
        if conf < min_conf or not text:
            continue

        suggestions.append({"text": text, "score": round(conf, 4)})

    # Deduplicate, sort by score
    seen, unique = set(), []
    for s in sorted(suggestions, key=lambda x: -x["score"]):
        if s["text"] not in seen:
            seen.add(s["text"])
            unique.append(s)

    return unique


# ── Quick test ────────────────────────────────────────────────────────────────
tok_g, mdl_g = load_finetuned()
results = phrase_generator(
    "Subject: Follow Up\n\nHi John, just wanted to follow up on", tok_g, mdl_g
)
for r in results:
    print(f"  [{r['score']:.3f}]  {r['text']}")

In [ ]:
PRONOUN_MAP = {
    "he ": "they ",
    "she ": "they ",
    "his ": "their ",
    "her ": "their ",
    "him ": "them ",
}
GENDERED_WORDS = {
    "chairman": "chairperson",
    "policeman": "police officer",
    "fireman": "firefighter",
    "stewardess": "flight attendant",
    "mankind": "humankind",
}
NSFW_WORDS = {"damn", "hell", "crap"}


def post_process(suggestion: str) -> str | None:
    """Apply pronoun neutralization, gendered word replacement, NSFW filter."""
    text = suggestion.lower()

    # NSFW filter — drop entire suggestion
    for word in NSFW_WORDS:
        if word in text.split():
            return None

    # Pronoun neutralization
    for gendered, neutral in PRONOUN_MAP.items():
        text = text.replace(gendered, neutral)

    # Gender-neutral word replacement
    for gendered, neutral in GENDERED_WORDS.items():
        text = text.replace(gendered, neutral)

    return text.strip().capitalize()


def get_best_suggestion(prompt, tok, mdl):
    """Full Smart Compose pipeline: generate → filter → post-process."""
    candidates = phrase_generator(prompt, tok, mdl)
    for c in candidates:
        cleaned = post_process(c["text"])
        if cleaned:
            return {"text": cleaned, "score": c["score"]}
    return None


# Test end-to-end
result = get_best_suggestion(
    "Subject: Meeting\n\nHi Sarah, I wanted to schedule a meeting to discuss",
    tok_g,
    mdl_g,
)
print("Best suggestion:", result)

In [ ]:
import gradio as gr

CSS = """
.suggestion-box { border-left: 3px solid #6366f1; padding: 8px 12px;
                  background: #f8f7ff; border-radius: 6px; margin: 4px 0; }
.conf-badge      { font-size: 0.75rem; color: #6b7280; margin-left: 8px; }
footer           { display: none !important; }
"""


def run_suggestion(subject, body):
    """Called on every keystroke (debounced) or button click."""
    if not body or len(body.split()) < 3:
        return (
            gr.update(visible=False),  # suggestion_row
            "",  # sug_1 text
            "",
            "",
            gr.update(visible=False),  # insert buttons
            gr.update(visible=False),
            gr.update(visible=False),
        )

    prompt = f"Subject: {subject}\n\n{body}" if subject.strip() else body
    candidates = phrase_generator(prompt, tok_g, mdl_g)

    texts, visibilities = [], []
    for i in range(3):
        if i < len(candidates):
            cleaned = post_process(candidates[i]["text"])
            conf = candidates[i]["score"]
            if cleaned:
                texts.append(f"{cleaned}   `conf: {conf:.3f}`")
                visibilities.append(gr.update(visible=True))
            else:
                texts.append("")
                visibilities.append(gr.update(visible=False))
        else:
            texts.append("")
            visibilities.append(gr.update(visible=False))

    row_visible = any(t != "" for t in texts)
    return (
        gr.update(visible=row_visible),
        texts[0],
        texts[1],
        texts[2],
        visibilities[0],
        visibilities[1],
        visibilities[2],
    )


def insert_suggestion(body, sug_text):
    """Strip the conf badge, append plain suggestion text to body."""
    plain = sug_text.split("`conf:")[0].strip()
    if plain:
        return body.rstrip() + " " + plain
    return body


def clear_all():
    return "", "", "", gr.update(visible=False), "", "", ""


with gr.Blocks(css=CSS, title="✉️ Smart Compose") as demo:
    gr.Markdown(
        "# ✉️ Smart Compose — Local\n`distilgpt2` fine-tuned on AESLC · Beam Search · Runs offline"
    )

    with gr.Tabs():
        with gr.TabItem("✏️ Compose"):
            with gr.Row():
                subject_box = gr.Textbox(
                    label="Subject", placeholder="e.g. Follow Up on Project", scale=3
                )
                with gr.Column(scale=1, min_width=120):
                    suggest_btn = gr.Button("✨ Suggest", variant="primary")
                    clear_btn = gr.Button("🗑 Clear", variant="secondary")

            body_box = gr.Textbox(
                label="Body",
                placeholder="Start typing your email…",
                lines=10,
                show_copy_button=True,
            )

            with gr.Column(visible=False) as suggestion_row:
                gr.Markdown("### 💡 Suggestions")
                with gr.Row():
                    sug_1 = gr.Textbox(
                        label="Option 1",
                        interactive=False,
                        elem_classes="suggestion-box",
                        scale=5,
                    )
                    ins_1 = gr.Button("Insert ➕", scale=1, visible=False)
                with gr.Row():
                    sug_2 = gr.Textbox(
                        label="Option 2",
                        interactive=False,
                        elem_classes="suggestion-box",
                        scale=5,
                    )
                    ins_2 = gr.Button("Insert ➕", scale=1, visible=False)
                with gr.Row():
                    sug_3 = gr.Textbox(
                        label="Option 3",
                        interactive=False,
                        elem_classes="suggestion-box",
                        scale=5,
                    )
                    ins_3 = gr.Button("Insert ➕", scale=1, visible=False)

        with gr.TabItem("⚙️ Fine-Tune"):
            gr.Markdown(
                """
            ### Upload your own emails to personalize the model

            Format each email as one JSON object per line (`.jsonl`):
            ```json
            {"subject": "Meeting", "body": "Hi John, let's connect about the Q2 roadmap…"}
            {"subject": "Follow Up", "body": "Just circling back on my previous message…"}
            ```

            After uploading, **re-run Cells 3–5** in the notebook to fine-tune.
            """
            )
            upload_box = gr.File(label="Upload .jsonl", file_types=[".jsonl"])
            upload_status = gr.Markdown(visible=False)

            def handle_upload(file):
                if file is None:
                    return gr.update(visible=False)
                import shutil

                shutil.copy(file.name, "my_emails.jsonl")
                return gr.update(
                    value="✅ **Saved as `my_emails.jsonl`** — re-run notebook Cells 3–5 to fine-tune!",
                    visible=True,
                )

            upload_box.change(handle_upload, inputs=upload_box, outputs=upload_status)

        with gr.TabItem("📊 Model Info"):
            gr.Markdown(
                f"""
            | Property | Value |
            |---|---|
            | Base model | `distilgpt2` (decoder-only Transformer) |
            | Fine-tune dataset | AESLC — Yale-LILY (~14K real Enron emails) |
            | Tokenizer | BPE subword tokenization (~50K vocab) |
            | Sampling strategy | Beam search (width = {BEAM_WIDTH}) |
            | Confidence filter | Drop suggestions below `{MIN_CONF}` |
            | Max suggestion length | `{MAX_SUGGESTION_WORDS}` words |
            | Device | `{DEVICE}` |

            **Evaluation metrics tracked during training:**
            - 📉 **Perplexity** — measures how well the model predicts held-out email text
            - 🎯 **ExactMatch@N** — % of beam completions matching first N words of ground truth
            """
            )

    suggestion_outputs = [
        suggestion_row,
        sug_1,
        sug_2,
        sug_3,
        ins_1,
        ins_2,
        ins_3,
    ]

    body_box.change(
        fn=run_suggestion,
        inputs=[subject_box, body_box],
        outputs=suggestion_outputs,
        show_progress="hidden",
    )

    suggest_btn.click(
        fn=run_suggestion,
        inputs=[subject_box, body_box],
        outputs=suggestion_outputs,
    )

    ins_1.click(fn=insert_suggestion, inputs=[body_box, sug_1], outputs=body_box)
    ins_2.click(fn=insert_suggestion, inputs=[body_box, sug_2], outputs=body_box)
    ins_3.click(fn=insert_suggestion, inputs=[body_box, sug_3], outputs=body_box)

    clear_btn.click(
        fn=clear_all,
        inputs=[],
        outputs=[
            subject_box,
            body_box,
            upload_status,
            suggestion_row,
            sug_1,
            sug_2,
            sug_3,
        ],
    )

demo.launch(
    inline=True,  # renders as an iframe directly in the notebook cell
    share=False,  # set True for a public Gradio tunnel URL
    server_port=7860,
    show_error=True,
)